# NB 05 — CatBoost Optuna (50 trials)

3-fold TimeSeriesSplit search → refit on full train → test eval → 5-fold OOF.

In [ ]:
import os, sys, time, warnings, json
from pathlib import Path
warnings.filterwarnings("ignore")

ROOT = Path.cwd()
if (ROOT / "src").is_dir():
    sys.path.insert(0, str(ROOT))
else:
    ROOT = ROOT.parent
    sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd, optuna
from catboost import CatBoostClassifier
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from sklearn.metrics import roc_auc_score

from src.config import DATA_PROCESSED, MODELS, REPORTS, SEED, CAT_LOWCARD, CAT_HIGHCARD
from src.utils import seed_all
from src.models.optuna_helpers import (
    cv_score_auc, build_oof_predictions, persist_study, save_phase2_artifacts,
    hard_stop_check, N_SPLITS_CV,
)

seed_all(SEED)
NAME = "catboost"
FAMILY = "catboost"
CAT_ALL = list(CAT_LOWCARD) + list(CAT_HIGHCARD)
print("ROOT:", ROOT)


In [ ]:
df = pd.read_parquet(DATA_PROCESSED / "df_master_extended.parquet")
idx = np.load(DATA_PROCESSED / "split_indices.npz")
train_idx, test_idx = idx["train_idx"], idx["test_idx"]
assert len(train_idx) == 23862 and len(test_idx) == 6138

train_sorted = df.iloc[train_idx].sort_values("visit_date")
sorted_orig_idx = train_sorted.index.to_numpy()

drop = ["visit_date", "sale_within_7d"]
X_train = train_sorted.drop(columns=drop).reset_index(drop=True)
y_train = train_sorted["sale_within_7d"].reset_index(drop=True).astype(int)
X_test = df.iloc[test_idx].drop(columns=drop).reset_index(drop=True)
y_test = df.iloc[test_idx]["sale_within_7d"].reset_index(drop=True).astype(int)
print("X_train", X_train.shape, "X_test", X_test.shape)


In [ ]:
def make_cb(params=None, *, with_es=True):
    base = dict(
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=SEED,
        verbose=0,
        task_type="CPU",
        thread_count=-1,
        allow_writing_files=False,
    )
    if with_es:
        base.update(dict(od_type="Iter", od_wait=50))
    if params:
        base.update(params)
    return CatBoostClassifier(**base)


def suggest_params(trial):
    return dict(
        iterations=trial.suggest_int("iterations", 500, 2500),
        depth=trial.suggest_int("depth", 4, 10),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 1.0, 10.0, log=True),
        random_strength=trial.suggest_float("random_strength", 0.0, 10.0),
        bagging_temperature=trial.suggest_float("bagging_temperature", 0.0, 1.0),
        border_count=trial.suggest_int("border_count", 32, 255),
        min_data_in_leaf=trial.suggest_int("min_data_in_leaf", 1, 50),
    )


def objective(trial):
    params = suggest_params(trial)
    return cv_score_auc(
        lambda: make_cb(params),
        X_train, y_train, FAMILY,
        trial=trial, n_splits=N_SPLITS_CV,
    )


In [ ]:
study = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=SEED),
    pruner=MedianPruner(n_warmup_steps=10),
    study_name="catboost_phase2",
)
t0 = time.perf_counter()
study.optimize(objective, n_trials=50, show_progress_bar=False, gc_after_trial=True)
print(f"[study] {len(study.trials)} trials in {time.perf_counter()-t0:.1f}s")
print(f"[study] best CV AUC = {study.best_value:.4f}")
print(f"[study] best params = {study.best_params}")

persist_study(study, ROOT / "studies" / "catboost.pkl")


In [ ]:
cv_auc = float(study.best_value)
print(f"asserting cv_auc {cv_auc:.4f} >= 0.76 ...")
assert cv_auc >= 0.76, f"HARD STOP: CatBoost CV AUC {cv_auc:.4f} < 0.76"


In [ ]:
best_params = dict(study.best_params)
final_model = make_cb(best_params, with_es=False)  # no held-out set on full train
t0 = time.perf_counter()
final_model.fit(X_train, y_train, cat_features=CAT_ALL, verbose=0)
fit_s = time.perf_counter() - t0
print(f"[refit] full train fit in {fit_s:.1f}s")

test_proba = final_model.predict_proba(X_test)[:, 1]
test_auc = float(roc_auc_score(y_test, test_proba))
print(f"[test] AUC = {test_auc:.4f}")
hard_stop_check(cv_auc, test_auc, FAMILY, NAME)


In [ ]:
t0 = time.perf_counter()
oof_proba, oof_pos_idx = build_oof_predictions(
    lambda: make_cb(best_params),
    X_train, y_train, FAMILY,
)
print(f"[oof] {len(oof_proba)} rows covered (of {len(X_train)}) in {time.perf_counter()-t0:.1f}s")
oof_auc = float(roc_auc_score(y_train.iloc[oof_pos_idx], oof_proba))
print(f"[oof] AUC = {oof_auc:.4f}")


In [ ]:
fi = final_model.get_feature_importance(prettified=True)
fi.columns = ["feature", "importance"]
top10 = fi.head(10).reset_index(drop=True)
top10.insert(0, "rank", np.arange(1, len(top10) + 1))
print(top10.to_string(index=False))


In [ ]:
extra = {"fit_seconds_final": round(fit_s, 1), "oof_auc": round(oof_auc, 4),
         "oof_rows_covered": int(len(oof_proba))}

paths = save_phase2_artifacts(
    name=NAME, family=FAMILY,
    best_estimator=final_model,
    sorted_orig_idx=sorted_orig_idx,
    oof_pos_idx=oof_pos_idx, oof_proba=oof_proba,
    test_index=test_idx, test_proba=test_proba,
    best_params=best_params, cv_auc=cv_auc, test_auc=test_auc,
    top10_importances=top10, extra=extra,
)
print(json.dumps(paths, indent=2))
